# Predição da Duração de Lesões no Futebol Europeu (Regressão)

**Objetivo:** prever `days_num` (dias de afastamento) a partir de informações disponíveis no momento do registro da lesão.

**Modelo planejado:** `RandomForestRegressor`

Este notebook foi reconstruído a partir do dataset original (`full_dataset_thesis - 1.csv`) e seguindo a abordagem metodológica apresentada no material mais recente (Etapa 3 revisada), com foco em:
- evitar vazamento de informação (data leakage);
- transformar o alvo (`log1p`) para lidar com assimetria/outliers;
- pipeline reprodutível com `ColumnTransformer` + `OneHotEncoder`;
- avaliação com métricas apropriadas de regressão (MAE/RMSE/R²);
- validação cruzada e ajuste de hiperparâmetros.


In [ ]:
# 1) Imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.feature_selection import mutual_info_regression
import joblib

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2) Carregamento do dataset

Arquivo esperado: `full_dataset_thesis - 1.csv`


In [ ]:
df = pd.read_csv('full_dataset_thesis - 2.csv')
print('Shape:', df.shape)
display(df.head())
print('
Colunas:', list(df.columns))


## 3) Visão geral e valores ausentes


In [ ]:
display(df.info())
display(df.describe(include='all'))

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Colunas com valores ausentes:')
display(missing)

if len(missing) > 0:
    plt.figure(figsize=(12,4))
    missing.plot(kind='bar')
    plt.title('Valores ausentes por coluna')
    plt.ylabel('Quantidade')
    plt.show()


## 4) Construção do alvo (`days_num`) e checagens

A coluna `Days` vem como string (ex.: `"43 days"`). Vamos extrair o número e criar o alvo numérico `days_num`.

> **Observação:** as datas (`injury_from_parsed`, `injury_until_parsed`) serão convertidas para datetime apenas para checagens — **não serão usadas como features** para evitar vazamento de informação.


In [ ]:
# Target numérico
# Ex.: '43 days' -> 43
# (robusto a strings variadas; extrai o primeiro número)
df['days_num'] = pd.to_numeric(df['Days'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

# Datas para checagens (NÃO entram no modelo)
df['injury_from_parsed'] = pd.to_datetime(df['injury_from_parsed'], errors='coerce')
df['injury_until_parsed'] = pd.to_datetime(df['injury_until_parsed'], errors='coerce')

print('Missing days_num:', df['days_num'].isna().sum())
print('Dias negativos:', (df['days_num'] < 0).sum())

# Checagem simples de inconsistência temporal
incons = (df['injury_until_parsed'] < df['injury_from_parsed']).sum()
print('Inconsistências (fim < início):', incons)


## 5) Seleção de variáveis (sem vazamento)

Para prever a duração no momento do registro, removemos variáveis pós-evento e/ou derivadas do alvo:
- `Days` (origina `days_num`)
- `Games missed` (pós-evento; altamente correlacionado com duração)
- `injury_from_parsed`, `injury_until_parsed` (podem revelar duração)
- `player_name` (ID-like; pode gerar overfitting)

**Features usadas (presentes no dataset):**
- `player_age` (numérica)
- `player_position`, `club`, `league`, `Season`, `Injury` (categóricas)


In [ ]:
target = 'days_num'
features = ['player_age','player_position','club','league','Season','Injury']

# Dataset de ML
df_ml = df[features + [target]].copy()

# Limpeza: remover alvo ausente e valores inválidos
df_ml = df_ml.dropna(subset=[target])
df_ml = df_ml[df_ml[target] >= 0]

print('Shape df_ml:', df_ml.shape)
display(df_ml.head())


## 6) Distribuição do alvo e transformação logarítmica

A duração de lesões costuma ter assimetria forte (muitos casos curtos e poucos muito longos).
Aplicamos:

$$y = \log(1 + days\_num)$$

para reduzir impacto de outliers e melhorar estabilidade do treino.


In [ ]:
plt.figure(figsize=(12,4))
sns.histplot(df_ml[target], bins=60)
plt.title('Distribuição de days_num')
plt.xlabel('Dias')
plt.show()

# transformação
df_ml['days_log'] = np.log1p(df_ml[target])

plt.figure(figsize=(12,4))
sns.histplot(df_ml['days_log'], bins=60)
plt.title('Distribuição de days_log = log1p(days_num)')
plt.xlabel('log1p(dias)')
plt.show()


## 7) Engenharia de features (opcional, conforme abordagem)

### 7.1) Agrupamento de lesões
O dataset tem alta cardinalidade em `Injury`. Um agrupamento simples pode ajudar (e pode ser expandido).

> Implementação abaixo é **tolerante a variações** (case-insensitive e por palavras-chave).

### 7.2) Faixa etária
Criaremos `age_group` com bins simples.


In [ ]:
def map_injury_group(s: str) -> str:
    if pd.isna(s):
        return 'Outros'
    x = str(s).strip().lower()
    # grupos por keywords (pode ajustar conforme EDA)
    muscular_kw = ['hamstring', 'muscle', 'calf', 'thigh', 'adductor', 'groin', 'strain', 'torn muscle']
    ligament_kw = ['acl', 'cruciate', 'ligament', 'meniscus']
    articular_kw = ['ankle', 'knee', 'hip', 'shoulder', 'joint', 'cartilage']
    structural_kw = ['back', 'spine', 'fracture', 'stress fracture', 'metatarsal']
    illness_kw = ['virus', 'ill', 'flu', 'covid', 'corona', 'infection', 'fever']

    if any(k in x for k in muscular_kw):
        return 'Muscular'
    if any(k in x for k in ligament_kw):
        return 'Ligamentar'
    if any(k in x for k in articular_kw):
        return 'Articular'
    if any(k in x for k in structural_kw):
        return 'Estrutural'
    if any(k in x for k in illness_kw):
        return 'Doença'
    return 'Outros'

# novas colunas
work = df_ml.copy()
work['injury_group'] = work['Injury'].apply(map_injury_group)

bins = [15, 21, 28, 35, 45, 60]
labels = ['Jovem', 'Pico', 'Experiente', 'Veterano', '60+']
work['age_group'] = pd.cut(work['player_age'], bins=bins, labels=labels, include_lowest=True)

display(work[['player_age','age_group','Injury','injury_group']].head(10))
print('Injury_group counts:')
display(work['injury_group'].value_counts())


## 8) Remoção de outliers (IQR)

A abordagem de referência remove outliers por IQR antes do treino. Isso pode ajudar na estabilidade, mas também pode remover casos reais (lesões longas).

Por isso deixamos um **toggle** para comparar os dois cenários.


In [ ]:
REMOVE_OUTLIERS_IQR = True

df_model = work.copy()

if REMOVE_OUTLIERS_IQR:
    Q1 = df_model[target].quantile(0.25)
    Q3 = df_model[target].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = df_model.shape[0]
    df_model = df_model[(df_model[target] >= lower) & (df_model[target] <= upper)].copy()
    after = df_model.shape[0]

    print(f'Remoção IQR ativada: {before} -> {after} linhas (removidas {before-after})')
else:
    print('Remoção IQR desativada: usando todos os dados.')

# Definindo X e y (alvo em log)
feature_cols = ['player_age','player_position','club','league','Season','Injury','injury_group','age_group']
X = df_model[feature_cols]
y = df_model['days_log']

print('X shape:', X.shape)
print('y shape:', y.shape)


## 9) Separação treino/teste

A abordagem original usa `train_test_split`. Como melhoria, também oferecemos opção de split por jogador (grupo), mas **por padrão** deixamos o split simples para manter aderência ao plano apresentado.


In [ ]:
USE_GROUP_SPLIT = False  # se quiser reduzir risco de vazamento por jogador, mude para True

if USE_GROUP_SPLIT:
    # usa player_name apenas como grupo (não é feature)
    groups = df.loc[df_model.index, 'player_name'].astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print('Split: GroupShuffleSplit por player_name')
    print('Players train/test:', groups.iloc[train_idx].nunique(), groups.iloc[test_idx].nunique())
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print('Split: train_test_split simples')

print('Train/Test:', X_train.shape, X_test.shape)


## 10) Pipeline de pré-processamento

- Numéricas: imputação mediana
- Categóricas: imputação mais frequente + OneHotEncoder


In [ ]:
numeric_features = ['player_age']
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


## 11) Modelo baseline (Regressão Linear)

Baseline serve como referência simples para saber se modelos mais complexos realmente agregam valor.


In [ ]:
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

baseline_model.fit(X_train, y_train)

y_pred_baseline_log = baseline_model.predict(X_test)

# Voltar ao espaço original (dias)
y_pred_baseline = np.expm1(y_pred_baseline_log)
y_test_original = np.expm1(y_test)

baseline_r2 = r2_score(y_test_original, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_baseline))
baseline_mae = mean_absolute_error(y_test_original, y_pred_baseline)

print('Baseline Linear Regression')
print('R2:', baseline_r2)
print('RMSE:', baseline_rmse)
print('MAE:', baseline_mae)


## 12) Modelo principal — Random Forest Regressor


In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_log = rf_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_real = np.expm1(y_test)


## 13) Avaliação do modelo

Métricas em **dias** (espaço original):
- MAE
- RMSE
- R²
- MAPE (opcional)

> Nota de compatibilidade: usamos `np.sqrt(mean_squared_error(...))` para evitar problemas de versão do scikit-learn (argumento `squared=False`).


In [ ]:
r2 = r2_score(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
mae = mean_absolute_error(y_real, y_pred)

# cuidado com divisões por zero (y_real pode conter 0)
den = np.where(y_real == 0, 1, y_real)
mape = np.mean(np.abs((y_real - y_pred) / den)) * 100

print('Random Forest')
print('R2:', r2)
print('RMSE:', rmse)
print('MAE:', mae)
print('MAPE:', mape)

comparison = pd.DataFrame({
    'Modelo': ['Linear Regression (baseline)', 'Random Forest'],
    'R2': [baseline_r2, r2],
    'RMSE': [baseline_rmse, rmse],
    'MAE': [baseline_mae, mae]
})

display(comparison)


## 14) Validação cruzada

Usamos `cross_val_score` com `R²` sobre o dataset completo (após filtros e engenharia).


In [ ]:
cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=5,
    scoring='r2'
)

print('CV R2 scores:', cv_scores)
print('CV R2 mean:', cv_scores.mean())


## 15) Ajuste de hiperparâmetros (RandomizedSearchCV)


In [ ]:
param_grid = {
    'regressor__n_estimators': [200, 300, 500],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    rf_model,
    param_distributions=param_grid,
    n_iter=12,
    cv=3,
    scoring='r2',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_
print('Melhores parâmetros:', random_search.best_params_)


## 16) Avaliação final (modelo ajustado)


In [ ]:
final_pred_log = best_model.predict(X_test)
final_pred = np.expm1(final_pred_log)

final_r2 = r2_score(y_real, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_real, final_pred))
final_mae = mean_absolute_error(y_real, final_pred)

print('Modelo final (tuned)')
print('R2:', final_r2)
print('RMSE:', final_rmse)
print('MAE:', final_mae)


## 17) Real vs Previsto + Resíduos


In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_real, final_pred, alpha=0.25)
mn, mx = float(y_real.min()), float(y_real.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('Valor real (dias)')
plt.ylabel('Valor previsto (dias)')
plt.title('Real vs Previsto (modelo final)')
plt.show()

residuals = y_real - final_pred
plt.figure(figsize=(10,4))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Distribuição dos resíduos (real - previsto)')
plt.show()


## 18) Mutual Information (correto)

Aqui calculamos **Mutual Information** de forma apropriada (regressão), após o pré-processamento.

> Observação: devido ao One-Hot, o número de features expandidas pode ser grande. Para manter o notebook leve, podemos amostrar os dados.


In [ ]:
SAMPLE_FOR_MI = 4000

X_mi = X.copy()
y_mi = y.copy()

if len(X_mi) > SAMPLE_FOR_MI:
    idx = X_mi.sample(SAMPLE_FOR_MI, random_state=RANDOM_STATE).index
    X_mi = X_mi.loc[idx]
    y_mi = y_mi.loc[idx]

X_processed = preprocessor.fit_transform(X_mi)

# mutual_info_regression aceita matriz esparsa em muitas versões; se falhar, converta para array (pode ser pesado)
try:
    mi_scores = mutual_info_regression(X_processed, y_mi, random_state=RANDOM_STATE)
except TypeError:
    mi_scores = mutual_info_regression(X_processed.toarray(), y_mi, random_state=RANDOM_STATE)

print('MI calculado. Número de features expandidas:', len(mi_scores))
print('Primeiros 10 scores:', mi_scores[:10])


## 19) Importância das variáveis (Random Forest)

Duas opções comuns:
- `feature_importances_` (do modelo) — rápido, mas pode ter viés com one-hot e alta cardinalidade;
- Permutation importance — mais interpretável, porém mais lenta.

Aqui mostramos `feature_importances_` com nomes das features expandidas pelo One-Hot.


In [ ]:
regressor = best_model.named_steps['regressor']
pre = best_model.named_steps['preprocessor']

# nomes das features após o pré-processamento
num_names = numeric_features
cat_ohe = pre.named_transformers_['cat'].named_steps['onehot']
cat_names = list(cat_ohe.get_feature_names_out(categorical_features))
feature_names = num_names + cat_names

importances = pd.Series(regressor.feature_importances_, index=feature_names).sort_values(ascending=False)

display(importances.head(25).to_frame('importance'))

plt.figure(figsize=(10,4))
importances.head(25).sort_values().plot(kind='barh')
plt.title('Top 25 importâncias (feature_importances_)')
plt.tight_layout()
plt.show()


## 20) Comparação adicional: Gradient Boosting


In [ ]:
gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=RANDOM_STATE))
])

gb_model.fit(X_train, y_train)

gb_pred_log = gb_model.predict(X_test)
gb_pred = np.expm1(gb_pred_log)

gb_r2 = r2_score(y_real, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_real, gb_pred))
gb_mae = mean_absolute_error(y_real, gb_pred)

comparison2 = pd.DataFrame({
    'Modelo': ['Linear Regression', 'Random Forest (tuned)', 'Gradient Boosting'],
    'R2': [baseline_r2, final_r2, gb_r2],
    'RMSE': [baseline_rmse, final_rmse, gb_rmse],
    'MAE': [baseline_mae, final_mae, gb_mae]
})

display(comparison2)


## 21) Exportação do melhor modelo

Você pode escolher qual modelo salvar. Por padrão, salvaremos o `best_model` (Random Forest ajustado).


In [ ]:
MODEL_FILENAME = 'random_forest_lesoes.pkl'
joblib.dump(best_model, MODEL_FILENAME)
print('Modelo salvo em:', MODEL_FILENAME)


## 22) Conclusões e próximos passos

**O que este notebook entrega:**
- pipeline completo e reprodutível para regressão de dias de afastamento;
- avaliação com métricas em dias;
- baseline e comparação entre modelos;
- validação cruzada e tuning;
- análise de resíduos e importâncias.

**Próximos passos (recomendados):**
1. Refinar `injury_group` com base no Top-N de `Injury` (EDA).
2. Avaliar split temporal (ex.: treinar em temporadas antigas e testar em temporadas recentes).
3. Tratar alta cardinalidade de `club`/`Injury` (agrupamentos, encoding alternativo).
